In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data" / "LaBr3 spectrum 1kb").exists():
    ROOT = ROOT.parent

SPECTRUM_DIR = ROOT / "data" / "LaBr3 spectrum 1kb"
new_files = {
    "Ba133": SPECTRUM_DIR / "LaBr3_Ba133_5mC.txt",
    "Co60":  SPECTRUM_DIR / "LaBr3_Co60_5mC.txt",
    "Cs137": SPECTRUM_DIR / "LaBr3_Cs137_5mC.txt",
    "Eu152": SPECTRUM_DIR / "LaBr3_Eu152_5mC.txt",
    "Na22":  SPECTRUM_DIR / "LaBr3_Na22_5mC.txt",
}

results = []
for isotope, path in new_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {isotope} spectrum: {path}")

    df = pd.read_csv(path, sep=r"\s+", names=["Energy_keV", "Counts"])
    e = df["Energy_keV"].to_numpy()
    c = df["Counts"].to_numpy()
    steps = np.diff(e)
    is_uniform = np.allclose(steps, steps[0], atol=1e-6)
    results.append({
        "isotope": isotope,
        "min_keV": e.min(), "max_keV": e.max(),
        "step_keV": steps[0] if is_uniform else "NON-UNIFORM",
        "n_bins": len(e), "total_counts": c.sum(),
    })

summary = pd.DataFrame(results)
print(summary)
print("\nH_d grid for comparison: min=0.5, max=6099.5, step=1.0, n_bins=6100")